# Assignment: Component-Level Evaluation — Hiring Assistant Search Step

**Goal:** Evaluate the web search component of a hiring assistant agent.  
We are given a pre-built mock search result. Our job is to write an evaluation function that checks whether the returned URLs come from trusted domains.

## Setup

In [2]:
import re
import utils

## Mock Search Result (Pre-built — do not change)

This is the output from the hiring assistant's web search step.  
It contains 6 URLs — some from trusted sources, some not.

In [3]:
mock_search_output = """
Here are recent findings on software engineer hiring trends in 2024:

1. The US Bureau of Labor Statistics projects strong demand:
   https://bls.gov/ooh/computer-and-information-technology/software-developers.htm

2. Stack Overflow's annual developer survey covers compensation:
   https://stackoverflow.com/insights/survey/2024

3. A detailed salary breakdown by role:
   https://randomblog.io/tech-salaries-2024

4. McKinsey report on the future of tech work:
   https://mckinsey.com/capabilities/people-and-organizational-performance/our-insights

5. An informal Reddit thread with anecdotal data:
   https://reddit.com/r/cscareerquestions/comments/abc123

6. World Economic Forum on tech skills gap:
   https://weforum.org/agenda/2024/01/tech-skills-gap
"""

utils.print_html(mock_search_output, title="Mock Search Output")

## Part 1 — Define Preferred Domains

We define a set of domains we trust for hiring and labor market data.  
Added 3 extra domains beyond the base list, each with a justification.

In [4]:
HIRING_TOP_DOMAINS = {
    # Base list
    "bls.gov",           # US Bureau of Labor Statistics — official government data
    "oecd.org",          # OECD — international economic and employment research
    "ilo.org",           # International Labour Organization — global labor standards
    "levels.fyi",        # Crowdsourced verified salary data for tech roles
    "stackoverflow.com", # Annual developer survey — large, well-documented sample
    "hbr.org",           # Harvard Business Review — peer-reviewed management research
    "mckinsey.com",      # McKinsey Global Institute — cited industry research
    "weforum.org",       # World Economic Forum — global labor market reports
    "arxiv.org",         # Academic preprints — CS and economics papers

    # Added domains (3 extra)
    "linkedin.com",      # LinkedIn Workforce Report — large-scale real hiring data
    "glassdoor.com",     # Glassdoor — employer reviews and verified salary surveys
    "nber.org",          # National Bureau of Economic Research — peer-reviewed economics
}

## Part 2 — Write the Evaluation Function

The function:
1. Extracts all URLs from the raw text with a regex.
2. Checks each URL's domain against `top_domains`.
3. Computes `ratio = preferred_count / total`.
4. Returns `(flag: bool, report: str)` — `True` = PASS.
5. Handles the edge case of no URLs found.

In [5]:
def evaluate_hiring_sources(top_domains: set, raw: str, min_ratio: float = 0.4):
    """
    Evaluate whether search results come from preferred/trusted domains.

    Args:
        top_domains: Set of trusted domain strings (e.g. 'bls.gov').
        raw:         Plain text containing URLs (e.g. agent search output).
        min_ratio:   Minimum fraction of preferred URLs required to PASS.

    Returns:
        (flag: bool, report: str)
    """
    # Step 1 — extract URLs
    url_pattern = re.compile(r'https?://[^\s\]\)>\}]+', flags=re.IGNORECASE)
    urls = url_pattern.findall(raw)

    # Edge case: no URLs found
    if not urls:
        return False, """### Evaluation — Hiring Source Quality
No URLs detected in the provided text.
Please make sure the search output contains links.
"""

    # Step 2 — check each URL against preferred domains
    total = len(urls)
    preferred_count = 0
    details = []

    for url in urls:
        domain = url.split("/")[2]  # extract hostname from URL
        is_preferred = any(td in domain for td in top_domains)
        if is_preferred:
            preferred_count += 1
        label = "✅ PREFERRED" if is_preferred else "❌ NOT PREFERRED"
        details.append(f"- {url} → {label}")

    # Step 3 — compute ratio
    ratio = preferred_count / total

    # Step 4 — PASS/FAIL flag
    flag = ratio >= min_ratio

    # Step 5 — build report
    report = f"""### Evaluation — Hiring Source Quality
- Total results:     {total}
- Preferred results: {preferred_count}
- Ratio:             {ratio:.2%}
- Threshold:         {min_ratio:.0%}
- Status:            {"✅ PASS" if flag else "❌ FAIL"}

**Details:**
{chr(10).join(details)}
"""
    return flag, report

## Part 3 — Run It on the Mock Data

Expected: **4 preferred out of 6** (bls.gov, stackoverflow.com, mckinsey.com, weforum.org) → ratio 66.67% → **PASS** at threshold 40%.

In [6]:
flag, report = evaluate_hiring_sources(HIRING_TOP_DOMAINS, mock_search_output)

print("Flag (True=PASS):", flag)
utils.print_html(report, title="Evaluation Summary")

Flag (True=PASS): True


## Part 4 — Experiments

We change one variable at a time and observe how the result changes.

In [7]:
# Experiment A — raise min_ratio to 0.8
# 4/6 = 66.67% is below 80%, so this should FAIL even though sources are the same.
flag_a, report_a = evaluate_hiring_sources(HIRING_TOP_DOMAINS, mock_search_output, min_ratio=0.8)
utils.print_html(report_a, title="Experiment A — min_ratio=0.8")

In [8]:
# Experiment B — remove mckinsey.com from the domain set
# Preferred drops from 4 to 3 → ratio 50% → still PASS at 40%, but weaker signal.
domains_b = HIRING_TOP_DOMAINS - {"mckinsey.com"}
flag_b, report_b = evaluate_hiring_sources(domains_b, mock_search_output)
utils.print_html(report_b, title="Experiment B — mckinsey.com removed")

In [9]:
# Experiment C — add reddit.com to the domain set
# reddit.com now counts as preferred → ratio rises to 5/6 = 83%.
# But this inflates the score: Reddit is user-generated, not a credible data source
# for hiring decisions. Adding it defeats the purpose of the eval.
domains_c = HIRING_TOP_DOMAINS | {"reddit.com"}
flag_c, report_c = evaluate_hiring_sources(domains_c, mock_search_output)
utils.print_html(report_c, title="Experiment C — reddit.com added")

## Part 5 — Short Answer Questions

**Q1. Why evaluate the search step alone instead of running the full pipeline every time?**  
Running the full pipeline (search → reflect → report) is expensive and noisy. Errors introduced in the search step get mixed with randomness from later steps, making it hard to tell what actually improved. Evaluating only the search step gives a clean, direct signal about that component's quality.

---

**Q2. What makes this evaluation "objective"? What is the ground truth?**  
It is objective because each URL either belongs to `HIRING_TOP_DOMAINS` or it does not — there is no judgment call. The ground truth is the domain allowlist itself: a predefined, explicit set of sources considered trustworthy for this query category.

---

**Q3. The eval fails. Name two fixes you could apply to the search step only.**  
1. **Refine the search prompt** — instruct the agent to prefer academic, government, or institutional sources explicitly (e.g., *"Prioritize results from .gov, .edu, or established research organizations"*).
2. **Filter tool results before returning** — post-process the raw Tavily/search output to drop URLs whose domain is not in `HIRING_TOP_DOMAINS` before the result is passed downstream.

---

**Q4. A teammate says: "Just set `min_ratio=0.0` so it always passes." What is wrong with this?**  
Setting `min_ratio=0.0` makes the evaluation a no-op — it always passes regardless of source quality. The eval then provides zero signal. Unreliable sources will silently flow into the reflection and report steps, corrupting the final output. The whole point of the evaluation is to catch bad sources early; removing the threshold eliminates that safeguard entirely.